# 30-NN统一重新评分

本入口仅计算30-NN。固定原权重、合成矩阵及真实参照PCA空间，对全量、九个子集和全量消融逐方法重新计算全部五项指标。全量和消融的UMAP同步使用30近邻。A1/A2直接从H5AD重新计算，A3在已经冻结并核验来源的PCA空间重新计算。结果与同口径既有结果逐项核对，不重新训练或选择模型。

修改下一单元格中的MODE与DATASET可选择任务；服务器运行使用simscdnam内核。

In [ ]:
from pathlib import Path
from datetime import datetime,timezone,timedelta
import sys,os,json,hashlib,shutil,time,gc
import nbformat,yaml
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from threadpoolctl import threadpool_limits
from IPython.display import display

PROJECT=Path('/home/sccasimp/program/xb/simscDNAm/releases/v1_knn30_20261117')
ROOT=Path('/home/sccasimp/data/xb/outputs/simscDNAm')
RUN=ROOT/'simscDNAm_knn30_20261117'
MODE='subset'
DATASET='liu2023/GSE213221'
RELATIVE=DATASET if MODE!='ablation' else 'ablation/liu2023/full'
directory=RUN/RELATIVE
directory.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(PROJECT))
os.chdir(PROJECT)
from evaluate.metrics import a1_metrics,paired_pearson,milisi
from evaluate.preprocessing import dense_block,region_pairs,read_labels,validate_region_axis,validate_real_duplicates
assert Path(sys.prefix).name.lower()=='simscdnam'
START=time.monotonic()

def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')

def progress(stage,**extra):
    record={'阶段':stage,'模式':MODE,'数据集':DATASET,'近邻数':30,
        '北京时间':datetime.now(timezone(timedelta(hours=8))).isoformat(),
        '已用秒数':round(time.monotonic()-START,1),**extra}
    write_json(directory/'progress.json',record)
    print(record,flush=True)

def memory_data(path):
    item=ad.read_h5ad(path)
    if sp.issparse(item.X):item.X=item.X.toarray()
    return item

progress('核对输入和冻结PCA空间')


In [ ]:
source=ROOT/'simscDNAm_knn30_20261111'/RELATIVE
prior=json.loads((source/'verification.json').read_text())
assert prior['状态']=='通过' and prior['近邻数']==30
input_records=dict(prior['输入文件状态'])
configuration=yaml.safe_load((source/'config.yaml').read_text())
if MODE=='subset':
    paths={k:Path(v) for k,v in configuration['输入'].items()}
    methods=['simscDNAm','FittedMarginal','simATAC','simCAS']
    pair_path=ROOT/'milisi_frozen_vae_20261104'/DATASET/'region_pairs.csv'
else:
    paths={'训练参照':Path(configuration['reference_h5ad']),'真实数据':Path(configuration['real_h5ad']),
           **{k:Path(v) for k,v in configuration['synthetic_h5ad'].items()}}
    methods=list(configuration['synthetic_h5ad'])
    pair_path=Path(configuration['region_pairs_csv'])
for path in [*paths.values(),pair_path,source/'scores.csv',source/'A3_PCA_coordinates.csv',
             source/'A3_type_details.csv',source/'A3_selected_regions.csv']:
    stat=path.stat()
    input_records.setdefault(str(path),[stat.st_size,stat.st_mtime_ns])
for path,before in input_records.items():
    stat=Path(path).stat()
    assert [stat.st_size,stat.st_mtime_ns]==before,path

coordinates=pd.read_csv(source/'A3_PCA_coordinates.csv')
pc_columns=[c for c in coordinates if c.startswith('PC')]
representations={};labels={};ids={}
for name,frame in coordinates.groupby('数据集',sort=False):
    representations[name]=frame[pc_columns].to_numpy()
    labels[name]=frame['细胞类型'].astype(str).to_numpy()
    ids[name]=frame['细胞ID'].to_numpy()
selected=pd.read_csv(source/'A3_selected_regions.csv')['原region索引'].to_numpy()
expected_scores=pd.read_csv(source/'scores.csv').set_index('方法').loc[methods]
expected_types=pd.read_csv(source/'A3_type_details.csv')
assert expected_scores['近邻数'].eq(30).all()
shutil.copy2(source/'A3_PCA_coordinates.csv',directory/'A3_PCA_coordinates.csv')
shutil.copy2(source/'A3_selected_regions.csv',directory/'A3_selected_regions.csv')

configuration.update({'milisi_neighbors':30,'output_dir':str(directory)})
if MODE!='subset':
    configuration['a1_regions_csv']=str(PROJECT/'evaluate/a1_marker_regions_liu2023.csv')
    (directory/'figures').mkdir(exist_ok=True)
    for stem in ['A1_marginal_regions','A1_region_by_type','A2_duplicate_Pearson']:
        for suffix in ['png','pdf']:shutil.copy2(source/'figures'/f'{stem}.{suffix}',directory/'figures'/f'{stem}.{suffix}')
    for name in ['input_diagnostics.csv','A1_plot_regions.csv','A1_plot_histograms.csv','A1_type_histograms.csv']:
        if (source/name).exists():shutil.copy2(source/name,directory/name)
CONFIG_PATH=directory/'config.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(configuration,allow_unicode=True,sort_keys=False),encoding='utf-8')
progress('读取真实留出矩阵')
real=memory_data(paths['真实数据'])
rows=np.arange(real.n_obs)
natural=real.n_vars-1000
assert natural==prior['自然区域数'] and real.n_obs==prior['留出细胞数']
np.testing.assert_array_equal(real.obs_names.to_numpy(),ids['真实数据'])
np.testing.assert_array_equal(read_labels(real,rows),labels['真实数据'])
pairs=region_pairs(real,pair_path=pair_path)
validate_real_duplicates(real,pairs)


In [ ]:
records=[];a1_types=[];a2_pairs=[];a2_summaries=[];a3_cells=[];a3_types=[];checks=[]
columns=['加权 1-WD','加权 MSE','加权 Pearson','A2_Pearson均值','miLISI']
with threadpool_limits(limits=8):
    for name in methods:
        progress('重新计算A1、A2及30近邻A3',方法=name)
        item=memory_data(paths[name])
        validate_region_axis(real,item)
        assert item.n_obs==real.n_obs
        np.testing.assert_array_equal(item.obs_names.to_numpy(),ids[name])
        np.testing.assert_array_equal(read_labels(item,rows),labels[name])
        a1,details=a1_metrics(real,item,rows,rows,natural)
        correlations=paired_pearson(dense_block(item,rows,pairs['原region索引'].to_numpy()),
                                    dense_block(item,rows,pairs['复制region索引'].to_numpy()))
        score,cells,groups=milisi(representations['真实数据'],labels['真实数据'],
                                 representations[name],labels[name],n_neighbors=30)
        record={'方法':name,**a1,'A2_Pearson均值':float(correlations.Pearson.mean()),
                '有效配对数':int(correlations.Pearson.notna().sum()),'miLISI':score,
                '近邻数':30,'入选区域数':len(selected),'PCA维数':len(pc_columns)}
        np.testing.assert_allclose(np.array([record[c] for c in columns]),
            expected_scores.loc[name,columns].to_numpy(dtype=float),rtol=1e-7,atol=1e-9)
        expected=expected_types[expected_types['方法']==name].set_index('细胞类型')['miLISI中位数'].sort_index()
        actual=groups.set_index('细胞类型')['miLISI中位数'].sort_index()
        assert actual.index.equals(expected.index)
        np.testing.assert_allclose(actual,expected,rtol=1e-7,atol=1e-9)
        details.insert(0,'方法',name)
        a1_types.append(details)
        a2_pairs.append(pd.concat([pd.DataFrame({'方法':[name]*len(pairs)}),pairs,correlations],axis=1))
        a2_summaries.append({'方法':name,'配对数':len(pairs),'有效配对数':record['有效配对数'],
                            'Pearson均值':record['A2_Pearson均值'],'Pearson中位数':float(correlations.Pearson.median()),
                            '无效配对数':int(correlations.Pearson.isna().sum())})
        cells.insert(0,'细胞ID',np.r_[ids['真实数据'],ids[name]])
        cells.insert(0,'方法',name);groups.insert(0,'方法',name)
        a3_cells.append(cells);a3_types.append(groups);records.append(record)
        checks.append({'方法':name,'全部五项重算最大绝对差':float(np.max(np.abs(
            np.array([record[c] for c in columns])-expected_scores.loc[name,columns].to_numpy(dtype=float)))),
            'A3类型中位数最大绝对差':float(np.max(np.abs(actual-expected))),'近邻数':30})
        pd.DataFrame(records).to_csv(directory/'scores_progress.csv',index=False)
        del item
        gc.collect()
summary=pd.DataFrame(records)
assert np.isfinite(summary[columns].to_numpy()).all()
summary.to_csv(directory/'scores.csv',index=False)
summary.to_csv(directory/'A1_A2_A3_summary.csv',index=False)
summary[['方法','加权 1-WD','加权 MSE','加权 Pearson','有效细胞类型权重']].to_csv(directory/'A1_summary.csv',index=False)
pd.concat(a1_types,ignore_index=True).to_csv(directory/'A1_type_details.csv',index=False)
pd.concat(a2_pairs,ignore_index=True).to_csv(directory/'A2_duplicate_correlations.csv',index=False)
pd.DataFrame(a2_summaries).to_csv(directory/'A2_pearson_summary.csv',index=False)
summary[['方法','miLISI','近邻数','入选区域数','PCA维数']].to_csv(directory/'A3_miLISI.csv',index=False)
pd.concat(a3_cells,ignore_index=True).to_csv(directory/'A3_miLISI_cells.csv',index=False)
pd.concat(a3_types,ignore_index=True).to_csv(directory/'A3_type_details.csv',index=False)
pd.DataFrame(checks).to_csv(directory/'recalculation_checks.csv',index=False)
display(summary)
progress('五项指标全部重新计算完成')


In [ ]:
if MODE!='subset':
    evaluation=nbformat.read(PROJECT/'evaluate/A1A2A3.ipynb',as_version=4)
    exec(evaluation.cells[3].source,globals())
    exec(evaluation.cells[5].source,globals())
    if MODE=='ablation':
        PLOT_COLORS.update({'CVAE_raw':'#7A8BA5','CVAE_mask':'#3B73B9','CVAE_align':'#D57A24','simscDNAm':'#269D91'})
        original_plot_name=plot_name
        def plot_name(name):
            return {'CVAE_raw':'CVAE','CVAE_mask':'CVAE + mask','CVAE_align':'CVAE + alignment',
                    'simscDNAm':'simscDNAm (full)'}.get(name,original_plot_name(name))
    joint_tables=[]
    with threadpool_limits(limits=8):
        for name in methods:
            progress('重新计算30近邻UMAP',方法=name)
            combined=np.vstack([representations['真实数据'],representations[name]])
            xy=umap.UMAP(n_neighbors=30,random_state=2022,transform_seed=2022,n_jobs=1).fit_transform(combined)
            joint_tables.append(pd.DataFrame({'方法':name,
                '来源':np.r_[np.repeat('真实',len(ids['真实数据'])),np.repeat('合成',len(ids[name]))],
                '细胞ID':np.r_[ids['真实数据'],ids[name]],'UMAP1':xy[:,0],'UMAP2':xy[:,1]}))
    joint=pd.concat(joint_tables,ignore_index=True)
    joint.to_csv(directory/'A3_joint_coordinates.csv',index=False)
    plot_joint_umap(joint,summary,methods,directory,42)

for path,before in input_records.items():
    stat=Path(path).stat()
    assert [stat.st_size,stat.st_mtime_ns]==before,path
verification={'状态':'通过','数据集':DATASET,'模式':MODE,'近邻数':30,
    '训练细胞数':prior['训练细胞数'],'留出细胞数':real.n_obs,'自然区域数':natural,
    '全部五项指标实际重算':True,'对照同口径既有结果一致':True,
    'A3逐类型结果一致':True,'全程仅计算30近邻':True,
    '未重新训练或修改合成矩阵':True,'使用冻结训练参照PCA空间':True,
    '近邻排除中心细胞':True,'A3按类型中位数等权平均':True,
    '输入文件状态':input_records,
    '指标代码SHA256':hashlib.sha256((PROJECT/'evaluate/metrics.py').read_bytes()).hexdigest()}
write_json(directory/'verification.json',verification)
write_json(directory/'run_summary.json',{'状态':'完成','配置':configuration,'方法':methods,'核验':verification})
progress('完成',核验='通过')
